# 04 Cluster Zone Segments

Build lightweight zone clusters from the enriched zone feature set and export cluster labels for the frontend.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from analytics.src.cluster_zone_labels import cluster_zones


In [ ]:
root = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
zones_path = root / 'frontend' / 'src' / 'data' / 'zones.enriched.json'
insights_path = root / 'frontend' / 'src' / 'data' / 'business-insights.json'

zones = pd.DataFrame(json.loads(zones_path.read_text(encoding='utf-8')))
insights = pd.DataFrame(json.loads(insights_path.read_text(encoding='utf-8')))
zones.head()

In [ ]:
feature_frame = zones.merge(insights, left_on='id', right_on='zoneId', how='left')
feature_frame['footfall'] = feature_frame[['footfallMorning', 'footfallAfternoon', 'footfallEvening']].mean(axis=1)
feature_frame['competition'] = feature_frame[['competitionCoffee', 'competitionFood', 'competitionRetail']].mean(axis=1)
feature_frame['complementarity'] = feature_frame[['complementarityCoffee', 'complementarityFood', 'complementarityRetail']].mean(axis=1)
feature_frame['store_count'] = feature_frame['storeCount'].fillna(0)
feature_frame['context_count'] = feature_frame['businessContext'].map(len)
clustered = cluster_zones(feature_frame)
clustered[['name', 'cluster_id', 'cluster_label']].sort_values(['cluster_id', 'name'])

In [ ]:
output_path = root / 'frontend' / 'src' / 'data' / 'zone-clusters.json'
output_path.write_text(
    json.dumps(
        clustered[['id', 'name', 'cluster_id', 'cluster_label', 'cluster_summary']]
        .rename(columns={'id': 'zoneId'})
        .to_dict(orient='records'),
        indent=2,
    ),
    encoding='utf-8',
)
print(f'Wrote {output_path}')